In [1]:
import numpy as np
import pandas
import pm4py
from matplotlib import pyplot as plt
from sklearn.mixture import GaussianMixture
import scipy.stats as stats
import ot
import os
from tqdm import tqdm
import collections
import matplotlib.dates as md
import importlib
import pickle
import random
import math
import CRPS.CRPS as pscore
import datetime

pandas.set_option('display.max_columns', None)
#pandas.set_option('display.max_rows', None)


import sys
sys.path.append('../../TaskExecutionTimeMining/')
from drbart_parser import *
from event_log_transformer import *

#sys.path.append('../../Evaluation')
sys.path.append('../../Evaluation/')
from conduct_evaluation import ConductEvaluation
from advanced_evaluation.advanced_evaluation import SampleOutcomesAdvanced

get_pscores = lambda likelihoods : [pscore(likelihoods[1][i], likelihoods[2][k][3]).compute()[0] for i, k in enumerate(list(likelihoods[0].keys()))]

In [2]:
#model_name = 'bpic_2017_all_2'
model_name = 'helpdesk'

n_processes = 32

log_name = 'train'
with open('../transformed_event_logs/Helpdesk_'+log_name+'.pickle', 'rb') as f:
    test_event_log = pickle.load(f)


test_event_log['Case ID'] = test_event_log['Case ID'].astype(str)
test_event_log['case:concept:name'] = test_event_log['Case ID']
test_event_log['time:timestamp_start'] = test_event_log['Complete Timestamp_start']
test_event_log['time:timestamp_complete'] = test_event_log['Complete Timestamp_complete']


known_resources = ['Value 1', 'Value 10', 'Value 11', 'Value 12', 'Value 13', 'Value 14', 'Value 15', 'Value 16', 'Value 17', 'Value 18', 'Value 19', 'Value 2', 'Value 20', 'Value 21', 'Value 22', 'Value 3', 'Value 4', 'Value 5', 'Value 6', 'Value 7', 'Value 8', 'Value 9']
known_activities = ['Assign seriousness', 'Closed', 'Create SW anomaly', 'DUPLICATE', 'INVALID', 'Insert ticket', 'RESOLVED', 'Require upgrade', 'Resolve SW anomaly', 'Resolve ticket', 'Schedule intervention', 'Take in charge ticket', 'VERIFIED', 'Wait']

In [3]:
drbart_model_path = '../../../models/advanced/'+model_name+'/concept-name/'
evaluator_A = ConductEvaluation(drbart_model_path, SampleOutcomesAdvanced, {
                                                        'activity_key' : 'Activity_start',
                                                        'resource_key' : '',
                                                        'resources' : False,
                                                        'categorical_args' : ['concept_name'],
                                                        'continuous_args' : [],
                                                        'known_activities' : known_activities,
                                                        'known_resources' : known_resources
                                                    },
                                     test_event_log, n_processes=n_processes,
                                )
likelihoods_A = evaluator_A.sample_cases(False, True)

100%|██████████| 3661/3661 [00:16<00:00, 223.03it/s]


In [4]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-4.163078916965064168931509825')

In [5]:
np.mean(get_pscores(likelihoods_A))

np.float64(695058.6713658362)

In [ ]:
drbart_model_path = '../../../models/advanced/'+model_name+'/concept-name_resource/'
evaluator_A = ConductEvaluation(drbart_model_path, SampleOutcomesAdvanced, {
                                                        'activity_key' : 'Activity_start',
                                                        'resource_key' : 'Resource_start',
                                                        'categorical_args' : ['resource','concept_name'],
                                                        'continuous_args' : [],
                                                        'known_activities' : known_activities,
                                                        'known_resources' : known_resources,
                                                        'strict_parsing' : True
                                                    },
                                    
                                     test_event_log, n_processes=n_processes,
                                )
likelihoods_A = evaluator_A.sample_cases(False, False)

  0%|          | 0/3661 [00:00<?, ?it/s]

Case sampling error: {'Case ID': 'Case 1534', 'Activity_start': 'Assign seriousness', 'Resource_start': 'Value 2', 'Complete Timestamp_start': Timestamp('2010-01-13 13:09:31'), 'Variant_start': 'Variant 33', 'Variant index_start': 33, 'Variant.1_start': 'Variant 33', 'seriousness_start': 'Value 1', 'customer_start': 'Value 176', 'product_start': 'Value 3', 'responsible_section_start': 'Value 4', 'seriousness_2_start': 'Value 1', 'service_level_start': 'Value 2', 'service_type_start': 'Value 1', 'support_section_start': 'Value 3', 'workgroup_start': 'Value 3', 'id_start': 7168, 'Activity_complete': 'Take in charge ticket', 'Resource_complete': 'Value 2', 'Complete Timestamp_complete': Timestamp('2010-01-26 09:50:48'), 'Variant_complete': 'Variant 33', 'Variant index_complete': 33, 'Variant.1_complete': 'Variant 33', 'seriousness_complete': 'Value 1', 'customer_complete': 'Value 176', 'product_complete': 'Value 3', 'responsible_section_complete': 'Value 4', 'seriousness_2_complete': 'Val

In [5]:
drbart_model_path = '../../../models/advanced/'+model_name+'/concept-name_resource/'

with open(drbart_model_path + 'gate.pickle', 'rb') as f:
    gate = pickle.load(f)

In [14]:
gate[0]

'Activity_start'

In [13]:
next(filter(lambda e : 'Take in charge ticket' in e[1], enumerate(gate[2])))

(9, ['Take in charge ticket'])

In [ ]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-4.266157542593290560952227066')

In [ ]:
np.mean(get_pscores(likelihoods_A))

np.float64(779558.780208338)

In [ ]:
drbart_model_path = '../../../models/advanced/'+model_name+'/concept-name_resource_seconds-in-day/'
evaluator_A = ConductEvaluation(drbart_model_path, SampleOutcomesAdvanced, {
                                                        'activity_key' : 'Activity_start',
                                                        'resource_key' : 'Resource_start',
                                                        'categorical_args' : ['concept_name', 'resource'],
                                                        'continuous_args' : ['seconds_in_day'],
                                                        'known_activities' : known_activities,
                                                        'known_resources' : known_resources,
                                                        'strict_parsing' : True
                                                    },
                                     test_event_log, n_processes=n_processes,
                                )
likelihoods_A = evaluator_A.sample_cases(False, True)

In [ ]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-4.784839425289035536191502781')

In [ ]:
np.mean(get_pscores(likelihoods_A))

np.float64(1057142.328200553)

In [ ]:
drbart_model_path = '../../../models/advanced/'+model_name+'/concept-name_resource_seconds-in-day_day-of-week/'
evaluator_A = ConductEvaluation(drbart_model_path, SampleOutcomesAdvanced, {
                                                        'activity_key' : 'Activity_start',
                                                        'resource_key' : 'Resource_start',
                                                        'categorical_args' : ['resource', 'concept_name', 'day_of_week'],
                                                        'continuous_args' : ['seconds_in_day'],
                                                        'known_activities' : known_activities,
                                                        'known_resources' : known_resources,
                                                        'strict_parsing' : True
                                                    },
                                     test_event_log, n_processes=n_processes,
                                )
likelihoods_A = evaluator_A.sample_cases(False, True)

In [ ]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-4.631427722689402512572761603')

In [ ]:
np.mean(get_pscores(likelihoods_A))

np.float64(974189.2577515558)

In [ ]:
drbart_model_path = '../../../models/advanced/'+model_name+'/concept-name_resource_seconds-in-day_activity-count_resource_count/'
evaluator_A = ConductEvaluation(drbart_model_path, SampleOutcomesAdvanced, {
                                                        'activity_key' : 'Activity_start',
                                                        'resource_key' : 'Resource_start',
                                                        'categorical_args' : ['resource', 'concept_name',
                                                                              '(lambda activity_count, known_activities : [0 if activity not in activity_count else activity_count[activity] for activity in known_activities])(activity_count, self.known_activities)',
                                                                              '(lambda resource_count, known_resources : [0 if resource not in resource_count else resource_count[resource] for resource in known_resources])(resource_count, self.known_resources)'
                                                        ],
                                                        'continuous_args' : ['seconds_in_day'],
                                                        'known_activities' : known_activities,
                                                        'known_resources' : known_resources,
                                                        'strict_parsing' : True
                                                    },
                                     test_event_log, n_processes=n_processes,
                                )
likelihoods_A = evaluator_A.sample_cases(False, True)

In [ ]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-4.850444648906791432449345367')

In [ ]:
np.mean(get_pscores(likelihoods_A))

np.float64(1272725.905413575)

In [ ]:
drbart_model_path = '../../../models/advanced/'+model_name+'/concept-name_resource_seconds-in-day_activity-count/'
evaluator_A = ConductEvaluation(drbart_model_path, SampleOutcomesAdvanced, {
                                                        'activity_key' : 'Activity_start',
                                                        'resource_key' : 'Resource_start',
                                                        'categorical_args' : ['resource', 'concept_name',
                                                                              '(lambda activity_count, known_activities : [0 if activity not in activity_count else activity_count[activity] for activity in known_activities])(activity_count, self.known_activities)',
                                                        ],
                                                        'continuous_args' : ['seconds_in_day'],
                                                        'known_activities' : known_activities,
                                                        'known_resources' : known_resources,
                                                        'strict_parsing' : True
                                                    },
                                     test_event_log, n_processes=n_processes,
                                )
likelihoods_A = evaluator_A.sample_cases(False, True)

In [ ]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-4.639781896351771593214952912')

In [ ]:
np.mean(get_pscores(likelihoods_A))

np.float64(1071337.9426214707)

In [ ]:
drbart_model_path = '../../../models/advanced/'+model_name+'/concept-name_resource_seconds-in-day_resource-count/'
evaluator_A = ConductEvaluation(drbart_model_path, SampleOutcomesAdvanced, {
                                                        'activity_key' : 'Activity_start',
                                                        'resource_key' : 'Resource_start',
                                                        'categorical_args' : ['resource', 'concept_name',
                                                                              '(lambda resource_count, known_resources : [0 if resource not in resource_count else resource_count[resource] for resource in known_resources])(resource_count, self.known_resources)'
                                                        ],
                                                        'continuous_args' : ['seconds_in_day'],
                                                        'known_activities' : known_activities,
                                                        'known_resources' : known_resources,
                                                        'strict_parsing' : rue
                                                    },
                                     test_event_log, n_processes=n_processes,
                                )
likelihoods_A = evaluator_A.sample_cases(False, True)

In [ ]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-4.825555741410616673973910565')

In [ ]:
np.mean(get_pscores(likelihoods_A))

np.float64(1201063.6170377082)